In [3]:
import pandas as pd
import requests
import re
import time

from bs4 import BeautifulSoup
from difflib import SequenceMatcher
from pathlib import Path
from urllib.parse import quote_plus

In [11]:
df = pd.read_excel(r"D:\Proyek FEB\Publikasi internasional.xlsx")
df.head()

,No,Title,Authors,Journal,SCOPUS Q,Tahun,Volume / Issue,LINK DOI/ARTIKEL,SCOPUS SITASI
0,1,Ethnic identity and internal migration decisio...,ILMIAWAN AUWALIN,Journal of Ethnic and Migration Studies,?,2019-01-11 00:00:00,"Vol. 14, Issue 13",10.1080/1369183X.2018.1561252,?
1,2,The role of service quality within Indonesian ...,BADRI MUNIR SUKOCO,Journal of Islamic Marketing,?,2020-01-14 00:00:00,"Vol. 11, Issue 1",10.1108/JIMA-03-2017-0033,?
2,3,Developing Islamic crowdfunding website platfo...,MUHAMAD NAFIK HADI RYANDONO,Journal of Islamic Marketing,?,2020-08-21 00:00:00,"Vol. 11, Issue 5",10.1108/JIMA-02-2019-0022,?
3,4,Developing Islamic crowdfunding website platfo...,ACHSANIA HENDRATMI,Journal of Islamic Marketing,?,2020-08-21 00:00:00,"Vol. 11, Issue 5",10.1108/JIMA-02-2019-0022,?
4,5,Developing Islamic crowdfunding website platfo...,PUJI SUCIA SUKMANINGRUM,Journal of Islamic Marketing,?,2020-08-21 00:00:00,"Vol. 11, Issue 5",10.1108/JIMA-02-2019-0022,?


In [12]:
def normalize_title(text):
    if pd.isna(text):
        return ""
    
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text.strip()

In [13]:
df["title_normalized"] = df["Title"].apply(normalize_title)

df[["Title", "title_normalized"]].head()

,Title,title_normalized
0,Ethnic identity and internal migration decisio...,ethnic identity and internal migration decisio...
1,The role of service quality within Indonesian ...,the role of service quality within indonesian ...
2,Developing Islamic crowdfunding website platfo...,developing islamic crowdfunding website platfo...
3,Developing Islamic crowdfunding website platfo...,developing islamic crowdfunding website platfo...
4,Developing Islamic crowdfunding website platfo...,developing islamic crowdfunding website platfo...


In [14]:
BASE_URL = "https://sinta.kemdiktisaintek.go.id/scopus"

headers = {
    "User-Agent": "Mozilla/5.0"
}

def search_sinta_scopus_by_title(title, sleep_time=1.5):
    query = quote_plus(str(title))
    url = f"{BASE_URL}?q={query}"
    
    response = requests.get(url, headers=headers, timeout=30)
    time.sleep(sleep_time)
    
    if response.status_code != 200:
        return {
            "status": "request_failed",
            "url": url,
            "http_status": response.status_code,
            "results": []
        }
    
    soup = BeautifulSoup(response.text, "html.parser")
    
    return {
        "status": "ok",
        "url": url,
        "html": response.text,
        "soup": soup
    }

In [15]:
sample_title = df.loc[0, "Title"]

result = search_sinta_scopus_by_title(sample_title)

result["status"], result["url"]

('ok',
 'https://sinta.kemdiktisaintek.go.id/scopus?q=Ethnic+identity+and+internal+migration+decision+in+Indonesia')

In [17]:
text = result["soup"].get_text(" ", strip=True)

text[:2000]

'SINTA - Science and Technology Index SINTA Author Subjects Affiliations Affiliations Departments Sources Journal Book IPR Researches Comunity Services Scopus Documents Check Quart. Scopus GS Documents FAQ WCU Registration Login Scopus Search Scopus Documents Results for "Ethnic identity and internal migration decision in Indonesia" clear search Previous 1 2 3 4 5 Next Page 1 of 10.000 | Total Records 100.000 Ethnic identity and internal migration decision in Indonesia Creator : Auwalin I. Journal of Ethnic and Migration Studies 2020 24 cited Q1 Journal Previous 1 2 3 4 5 Next Page 1 of 10.000 | Total Records 100.000 Get More with SINTA Insight Go to Insight Academic Rank Get More with SINTA Insight Go to Insight'

In [16]:
def extract_quartile_from_text(text):
    if not text:
        return None
    
    match = re.search(r"\b(Q[1-4])\s+Journal\b", text, flags=re.IGNORECASE)
    
    if match:
        return match.group(1).upper()
    
    if "no-Q" in text or "no q" in text.lower():
        return "no-Q"
    
    return None

In [18]:
extract_quartile_from_text(text)

'Q1'

In [19]:
def debug_sinta_links(soup):
    links = []
    
    for a in soup.find_all("a"):
        label = a.get_text(" ", strip=True)
        href = a.get("href")
        
        if label or href:
            links.append({
                "label": label,
                "href": href
            })
    
    return pd.DataFrame(links)

links_df = debug_sinta_links(result["soup"])
links_df.head(30)

,label,href
0,SINTA,https://sinta.kemdiktisaintek.go.id
1,,https://sinta.kemdiktisaintek.go.id
2,Author,https://sinta.kemdiktisaintek.go.id/authors
3,Subjects,https://sinta.kemdiktisaintek.go.id/subjects
4,Affiliations,#
5,Affiliations,https://sinta.kemdiktisaintek.go.id/affiliations
6,Departments,https://sinta.kemdiktisaintek.go.id/departments
7,Sources,#
8,Journal,https://sinta.kemdiktisaintek.go.id/journals
9,Book,https://sinta.kemdiktisaintek.go.id/books


In [20]:
def get_scopus_q_from_sinta_title(title):
    result = search_sinta_scopus_by_title(title)
    
    if result["status"] != "ok":
        return {
            "title": title,
            "scopus_q": None,
            "match_score": 0,
            "status": result["status"],
            "source_url": result.get("url")
        }
    
    soup = result["soup"]
    page_text = soup.get_text(" ", strip=True)
    
    score = similarity(title, page_text)
    q = extract_quartile_from_text(page_text)
    
    if q and score > 0.15:
        status = "found_possible"
    elif q:
        status = "q_found_but_low_match"
    else:
        status = "not_found"
    
    return {
        "title": title,
        "scopus_q": q,
        "match_score": score,
        "status": status,
        "source_url": result.get("url")
    }

In [21]:
test_titles = df["Title"].dropna().head(5).tolist()

test_results = []

for title in test_titles:
    test_results.append(get_scopus_q_from_sinta_title(title))

pd.DataFrame(test_results)

NameError: name 'similarity' is not defined